# Fine-tuning de modelos de lenguaje en español: segunda tanda

Segunda tanda de fine-tuning, con el mismo esquema que `finetune_titulos.ipynb` (BETO sobre el título: accuracy OOF 0,836, AUC 0,913). Corre varias **configuraciones una detrás de otra**, definidas en la lista `CONFIGURACIONES` de la primera celda de código:

| configuración | modelo | texto | max_length | semilla |
|---|---|---|---|---|
| a) `beto_titulo_warranty` | BETO uncased (`dccuchile/bert-base-spanish-wwm-uncased`) | título + `" [SEP] "` + warranty (`""` si no hay) | 96 | 7 |
| b) `beto_cased_titulo` | BETO cased (`dccuchile/bert-base-spanish-wwm-cased`); si no descarga, `bertin-project/bertin-roberta-base-spanish` (`bertin_titulo`) | título | 48 | 42 |

**Qué hace cada configuración** (igual que el notebook anterior):
1. Usa los 5 folds de `folds.csv`, los mismos de la validación del proyecto.
2. En cada fold ajusta el modelo con las otras 4 durante 3 épocas y se queda con la época de menor log loss en el fold de validación (early stopping).
3. Guarda cada fold apenas termina y al final arma `finetune_{nombre}_oof.csv` (fila, fold, proba_used) y `finetune_{nombre}_test.csv` (fila de test desde 0, proba_used) en `/kaggle/working`.
4. Muestra accuracy, AUC y log loss OOF en total y en vendedores nuevos con una publicación, comparados con BETO título y con el TF-IDF del proyecto. Si se subió `finetune_titulo_oof.csv` como dato de entrada, muestra también la correlación con BETO título.

**Retomar:** si una configuración ya tiene sus dos archivos finales completos en `/kaggle/working`, se saltea; dentro de una configuración, los folds ya guardados no se reentrenan. Si un modelo falla al descargar, se pasa al siguiente candidato de su lista.

**Cuánto tarda** (GPU T4): el tiempo crece más o menos con `max_length`.
- a) ~12-16 minutos por fold (textos de 96 tokens): **~60-80 minutos**;
- b) ~6-10 minutos por fold, como la primera corrida: **~35-50 minutos**;
- **total: ~1,5-2,5 horas**, muy por debajo del máximo de una sesión de Kaggle (~9 horas; la cuota semanal es de ~30 horas de GPU).

Al principio se muestra una estimación previa y, después del primer fold, una estimación con el tiempo medido. Si alguna de las dos supera `LIMITE_HORAS`, aparece un aviso.

In [ ]:
# ---- Parámetros ----
# Cada configuración prueba sus candidatos en orden y usa el primero que descarga. "nombre" define los archivos
# de salida: finetune_{nombre}_oof.csv y finetune_{nombre}_test.csv.
# "PlanTL-GOB-ES/roberta-base-bne" no sirve: el repositorio fue discontinuado y no tiene los pesos.
CONFIGURACIONES = [
    {"clave": "a", "texto": "titulo_warranty", "max_length": 96, "semilla": 7,
     "candidatos": [{"modelo": "dccuchile/bert-base-spanish-wwm-uncased", "nombre": "beto_titulo_warranty"}]},
    {"clave": "b", "texto": "titulo", "max_length": 48, "semilla": 42,
     "candidatos": [{"modelo": "dccuchile/bert-base-spanish-wwm-cased", "nombre": "beto_cased_titulo"},
                    {"modelo": "bertin-project/bertin-roberta-base-spanish", "nombre": "bertin_titulo"}]},
]
BATCH = 64  # si se queda sin memoria de GPU, bajarlo a 32
LEARNING_RATE = 3e-5
EPOCAS = 3
WARMUP = 0.10
WEIGHT_DECAY = 0.01
FP16 = True
RUTA_ENTRADA = "/kaggle/input"
RUTA_SALIDA = "/kaggle/working"
REQUERIR_GPU = True
MAX_FILAS = None  # solo para pruebas: limita las filas de train y test
LIMITE_HORAS = 8.5  # una sesión de Kaggle dura como máximo ~9 horas
MINUTOS_FOLD_REFERENCIA = 8  # minutos por fold con max_length 48 en una T4 (primera corrida: 6-10)

# Referencias (OOF con los mismos folds). Segmento: vendedor que no aparece en los otros folds y tiene una
# sola publicación en el suyo (la definición del proyecto, 19.926 filas).
SEGMENTO = "nuevo con una publicación"
REFERENCIAS = {
    "BETO título": {"total": {"accuracy": 0.8357, "auc": 0.9134, "log_loss": 0.3849},
                    SEGMENTO: {"accuracy": 0.7773, "auc": 0.8602, "log_loss": 0.4966}},
    "TF-IDF": {"total": {"accuracy": 0.8220, "auc": 0.8979, "log_loss": 0.4069},
               SEGMENTO: {"accuracy": 0.7561, "auc": 0.8401, "log_loss": 0.4995}},
}

In [ ]:
import glob
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

for cfg in CONFIGURACIONES:
    assert cfg["texto"] in ("titulo", "titulo_warranty"), 'texto tiene que ser "titulo" o "titulo_warranty"'
nombres = [c["nombre"] for cfg in CONFIGURACIONES for c in cfg["candidatos"]]
assert len(nombres) == len(set(nombres)), "Cada candidato tiene que tener un nombre distinto"
hay_gpu = torch.cuda.is_available()
print(f"torch {torch.__version__} | GPU disponible: {hay_gpu}"
      + (f" ({torch.cuda.get_device_name(0)})" if hay_gpu else ""))
if REQUERIR_GPU and not hay_gpu:
    raise RuntimeError("No hay GPU. En el panel derecho del notebook: Settings > Accelerator > GPU T4 x2 (o GPU T4), "
                       "y volvé a ejecutar. Sin GPU el entrenamiento tardaría muchas horas.")
DISPOSITIVO = torch.device("cuda" if hay_gpu else "cpu")
os.makedirs(RUTA_SALIDA, exist_ok=True)
USAR_FP16 = FP16 and hay_gpu
INICIO_CORRIDA = time.time()

## Datos

Se buscan los archivos con `glob` en cualquier subcarpeta de `/kaggle/input`. `finetune_titulo_oof.csv` (las predicciones OOF de BETO título, de la primera corrida) es opcional: si está, se calcula la correlación con cada configuración.

In [ ]:
def buscar(nombre, obligatorio=True):
    encontrados = sorted(glob.glob(os.path.join(RUTA_ENTRADA, "**", nombre), recursive=True))
    if not encontrados:
        if not obligatorio:
            return None
        contenido = sorted(glob.glob(os.path.join(RUTA_ENTRADA, "**", "*"), recursive=True))[:100]
        raise FileNotFoundError(
            f"No se encontró {nombre} dentro de {RUTA_ENTRADA}. Revisá que la competencia y el dataset con folds.csv "
            f"estén agregados como datos de entrada (panel derecho: Input > Add Input). Lo que hay ahora:\n"
            + ("\n".join(contenido) if contenido else "(vacío)"))
    if len(encontrados) > 1:
        print(f"Aviso: hay {len(encontrados)} archivos {nombre}; se usa {encontrados[0]}")
    return encontrados[0]


def leer_jsonlines(ruta, columnas):
    filas = []
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            if linea.strip():
                registro = json.loads(linea)
                filas.append({c: registro.get(c) for c in columnas})
                if MAX_FILAS and len(filas) >= MAX_FILAS:
                    break
    return pd.DataFrame(filas)


RUTA_TRAIN, RUTA_TEST, RUTA_FOLDS = buscar("train_data.jsonlines"), buscar("test_data.jsonlines"), buscar("folds.csv")
RUTA_BETO = buscar("finetune_titulo_oof.csv", obligatorio=False)
print(f"train: {RUTA_TRAIN}\ntest: {RUTA_TEST}\nfolds: {RUTA_FOLDS}\nBETO título (opcional): {RUTA_BETO or 'no está'}")
train = leer_jsonlines(RUTA_TRAIN, ["title", "warranty", "seller_id", "condition"])
test = leer_jsonlines(RUTA_TEST, ["title", "warranty", "seller_id"])
folds = pd.read_csv(RUTA_FOLDS)
if MAX_FILAS:
    folds = folds[folds["fila"] < len(train)]
assert list(folds.columns) == ["fila", "fold"], "folds.csv tiene que tener las columnas fila y fold"
assert len(folds) == len(train) and (folds["fila"].to_numpy() == np.arange(len(train))).all(), \
    "folds.csv no corresponde a train_data.jsonlines (distinta cantidad u orden de filas)"
y = (train["condition"] == "used").astype(int).to_numpy()
FOLD = folds["fold"].to_numpy()
print(f"train {len(train):,} filas ({y.mean():.1%} used) | test {len(test):,} filas | "
      f"filas por fold: {np.bincount(FOLD)[1:].tolist()}")

PROBA_BETO = None
if RUTA_BETO:
    beto = pd.read_csv(RUTA_BETO)
    beto = beto[beto["fila"] < len(train)].sort_values("fila")
    if (beto["fila"].to_numpy() == np.arange(len(train))).all() and (beto["fold"].to_numpy() == FOLD).all():
        PROBA_BETO = beto["proba_used"].to_numpy()
    else:
        print("Aviso: finetune_titulo_oof.csv no coincide con train y folds.csv (filas o folds distintos); se ignora.")

# Segmento: vendedor que no aparece en los otros folds y tiene una sola publicación en el suyo
vendedor = train["seller_id"].astype(str)
en_su_fold = pd.Series(1, index=vendedor.index).groupby([vendedor, FOLD]).transform("size")
en_otros_folds = vendedor.map(vendedor.value_counts()) - en_su_fold
MASCARAS = {"total": np.ones(len(y), dtype=bool), SEGMENTO: ((en_otros_folds == 0) & (en_su_fold == 1)).to_numpy()}
print(f"Filas en el segmento '{SEGMENTO}': {MASCARAS[SEGMENTO].sum():,}")

## Estado y estimación previa

Una configuración está **completa** si `finetune_{nombre}_oof.csv` y `finetune_{nombre}_test.csv` de alguno de sus candidatos existen en `/kaggle/working` con todas las filas. Las completas se saltean.

La estimación previa usa `MINUTOS_FOLD_REFERENCIA` (minutos por fold con 48 tokens en una T4) multiplicado por `max_length / 48`, porque el costo crece más o menos con el largo de los textos.

In [ ]:
def archivos_finales(nombre):
    return (os.path.join(RUTA_SALIDA, f"finetune_{nombre}_oof.csv"), os.path.join(RUTA_SALIDA, f"finetune_{nombre}_test.csv"))


def completa(nombre):
    archivo_oof, archivo_test = archivos_finales(nombre)
    if not (os.path.exists(archivo_oof) and os.path.exists(archivo_test)):
        return False
    try:
        oof, pred_test = pd.read_csv(archivo_oof), pd.read_csv(archivo_test)
    except Exception:
        return False
    return (len(oof) == len(train) and (np.sort(oof["fila"].to_numpy()) == np.arange(len(train))).all()
            and len(pred_test) == len(test) and oof["proba_used"].notna().all() and pred_test["proba_used"].notna().all())


def nombre_completo(cfg):
    return next((c["nombre"] for c in cfg["candidatos"] if completa(c["nombre"])), None)


def folds_guardados(nombre):
    return sum(all(os.path.exists(os.path.join(RUTA_SALIDA, f"{nombre}_fold_{k}_{s}")) for s in ("oof.csv", "test.csv", "info.json"))
               for k in range(1, 6))


def factor(cfg):
    return cfg["max_length"] / 48


def mostrar_estimacion(minutos_fold_48, titulo, folds_hechos=None):
    # Minutos que faltan: folds sin hacer de las configuraciones pendientes, escalados por max_length
    folds_hechos = folds_hechos or {}
    filas, total = [], 0.0
    for cfg in CONFIGURACIONES:
        if nombre_completo(cfg):
            filas.append({"configuración": cfg["clave"], "estado": "completa", "minutos": 0.0})
            continue
        faltan = 5 - folds_hechos.get(cfg["clave"], max(folds_guardados(c["nombre"]) for c in cfg["candidatos"]))
        minutos = faltan * minutos_fold_48 * factor(cfg)
        total += minutos
        filas.append({"configuración": cfg["clave"], "estado": f"faltan {faltan} folds", "minutos": round(minutos, 1)})
    display(pd.DataFrame(filas).set_index("configuración"))
    transcurrido = (time.time() - INICIO_CORRIDA) / 3600
    print(f"{titulo}: faltan ~{total / 60:.1f} horas (ya pasaron {transcurrido:.1f}); total estimado "
          f"~{transcurrido + total / 60:.1f} horas.")
    if transcurrido + total / 60 > LIMITE_HORAS:
        print(f"AVISO: la estimación supera {LIMITE_HORAS} horas y puede no entrar en una sesión de Kaggle (~9 horas). "
              f"Las configuraciones y folds terminados quedan guardados: descargalos y retomá en otra sesión, "
              f"o sacá configuraciones de CONFIGURACIONES.")


mostrar_estimacion(MINUTOS_FOLD_REFERENCIA, "Estimación previa (antes de entrenar)")

## Texto, modelo y entrenamiento

- **Texto:** el título original con los caracteres con problemas de codificación corregidos (`"ã¿"` → `"ó"`, `"ã\x8d"` → `"í"`). Con `texto = "titulo_warranty"` se agrega `" [SEP] "` y el texto de warranty (vacío si no hay). Se usa el separador del tokenizador (`[SEP]` en BETO; `</s>` en un RoBERTa).
- **Minúsculas:** solo si el nombre del modelo dice `uncased`. BETO *cased* y BERTIN reciben el texto tal cual.
- **Descarga:** antes de entrenar se descargan el tokenizador y los pesos. Si falla, se pasa al siguiente candidato; si fallan todos, la configuración se saltea y se sigue con la próxima.
- **Entrenamiento** (igual que la primera corrida): AdamW (`weight_decay` 0,01, sin aplicarlo a sesgos ni LayerNorm), learning rate 3e-5 con *warmup* lineal del 10% y bajada lineal, lotes de 64, 3 épocas, fp16, la semilla de la configuración. Al final de cada época se predice el fold de validación y test, y se guardan las predicciones de la época con menor log loss en validación.
- Cada fold se guarda como `{nombre}_fold_{k}_oof.csv`, `_test.csv` e `_info.json` apenas termina.

In [ ]:
REPARACIONES = {"ã¿": "ó", "ã\x8d": "í"}


def limpiar(texto):
    texto = texto if isinstance(texto, str) else ""
    for malo, bueno in REPARACIONES.items():
        texto = texto.replace(malo, bueno)
    return texto.strip()


def armar_texto(df, cfg, modelo, tokenizador):
    textos = df["title"].map(limpiar)
    if cfg["texto"] == "titulo_warranty":
        separador = tokenizador.sep_token or "[SEP]"
        textos = textos + f" {separador} " + df["warranty"].map(limpiar)
    if "uncased" in modelo.lower():
        textos = textos.str.lower()
    return textos.tolist()


def descargar(cfg):
    # Devuelve (candidato, tokenizador) del primer candidato que descarga, o (None, None)
    for candidato in cfg["candidatos"]:
        try:
            print(f"Descargando {candidato['modelo']}...", flush=True)
            tokenizador = AutoTokenizer.from_pretrained(candidato["modelo"])
            modelo = AutoModelForSequenceClassification.from_pretrained(candidato["modelo"], num_labels=2)
            del modelo
            return candidato, tokenizador
        except Exception as error:
            print(f"  Falló la descarga de {candidato['modelo']}: {type(error).__name__}: {str(error)[:300]}", flush=True)
    return None, None


def tokenizar(tokenizador, textos, max_length):
    codificado = tokenizador(textos, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")
    return codificado["input_ids"], codificado["attention_mask"]


def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


def predecir(modelo, ids, mascara, lote=256):
    modelo.eval()
    probas = []
    with torch.no_grad():
        for i in range(0, len(ids), lote):
            with torch.autocast(device_type=DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_FP16):
                logits = modelo(input_ids=ids[i:i + lote].to(DISPOSITIVO),
                                attention_mask=mascara[i:i + lote].to(DISPOSITIVO)).logits
            probas.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probas)


def entrenar_fold(k, nombre_modelo, semilla, ids_train, mascara_train, ids_test, mascara_test):
    fijar_semilla(semilla)
    idx_tr, idx_val = np.flatnonzero(FOLD != k), np.flatnonzero(FOLD == k)
    modelo = AutoModelForSequenceClassification.from_pretrained(nombre_modelo, num_labels=2).to(DISPOSITIVO)
    sin_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    grupos = [
        {"params": [p for n, p in modelo.named_parameters() if not any(s in n for s in sin_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in modelo.named_parameters() if any(s in n for s in sin_decay)], "weight_decay": 0.0},
    ]
    optimizador = torch.optim.AdamW(grupos, lr=LEARNING_RATE)
    pasos_por_epoca = int(np.ceil(len(idx_tr) / BATCH))
    programa = get_linear_schedule_with_warmup(optimizador, int(WARMUP * pasos_por_epoca * EPOCAS), pasos_por_epoca * EPOCAS)
    escalador = torch.amp.GradScaler("cuda", enabled=USAR_FP16)
    generador = torch.Generator().manual_seed(semilla)
    etiquetas = torch.tensor(y, dtype=torch.long)

    mejor = {"log_loss": np.inf}
    historia = []
    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        inicio = time.time()
        orden = idx_tr[torch.randperm(len(idx_tr), generator=generador).numpy()]
        perdida_total = 0.0
        for paso in range(pasos_por_epoca):
            lote = orden[paso * BATCH:(paso + 1) * BATCH]
            with torch.autocast(device_type=DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_FP16):
                salida = modelo(input_ids=ids_train[lote].to(DISPOSITIVO), attention_mask=mascara_train[lote].to(DISPOSITIVO),
                                labels=etiquetas[lote].to(DISPOSITIVO))
            optimizador.zero_grad(set_to_none=True)
            escalador.scale(salida.loss).backward()
            escalador.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            escalador.step(optimizador)
            escalador.update()
            programa.step()
            perdida_total += salida.loss.item()
        proba_val = predecir(modelo, ids_train[idx_val], mascara_train[idx_val])
        perdida_val = log_loss(y[idx_val], np.clip(proba_val, 1e-7, 1 - 1e-7), labels=[0, 1])
        historia.append({"época": epoca, "log loss train": perdida_total / pasos_por_epoca, "log loss val": perdida_val,
                         "accuracy val": accuracy_score(y[idx_val], proba_val >= 0.5), "minutos": (time.time() - inicio) / 60})
        print(f"  fold {k}, época {epoca}: log loss train {historia[-1]['log loss train']:.4f} | val {perdida_val:.4f} | "
              f"accuracy val {historia[-1]['accuracy val']:.4f} | {historia[-1]['minutos']:.1f} min", flush=True)
        if perdida_val < mejor["log_loss"]:
            mejor = {"log_loss": perdida_val, "epoca": epoca, "proba_val": proba_val,
                     "proba_test": predecir(modelo, ids_test, mascara_test)}
    del modelo, optimizador
    if hay_gpu:
        torch.cuda.empty_cache()
    return idx_val, mejor, historia

### Comparación de cada configuración

Accuracy, AUC y log loss OOF en total y en vendedores nuevos con una publicación, contra BETO título y el TF-IDF. Si está `finetune_titulo_oof.csv`, también la correlación con BETO título, el % de filas donde predicen distinto y el promedio simple de los dos (una idea de cuánto podrían sumar juntos).

In [ ]:
def metricas(proba, mascara):
    p = np.clip(proba[mascara], 1e-7, 1 - 1e-7)
    return {"accuracy": accuracy_score(y[mascara], p >= 0.5), "auc": roc_auc_score(y[mascara], p),
            "log_loss": log_loss(y[mascara], p, labels=[0, 1])}


def comparar(nombre, proba):
    filas = []
    for segmento, mascara in MASCARAS.items():
        propias = metricas(proba, mascara)
        for metrica in ("accuracy", "auc", "log_loss"):
            fila = {"segmento": segmento, "métrica": metrica, nombre: propias[metrica]}
            for referencia, valores in REFERENCIAS.items():
                fila[referencia] = valores[segmento][metrica]
            fila["diferencia con BETO título"] = propias[metrica] - valores_beto(segmento, metrica)
            if PROBA_BETO is not None:
                fila["promedio con BETO título"] = metricas((proba + PROBA_BETO) / 2, mascara)[metrica]
            filas.append(fila)
    display(pd.DataFrame(filas).set_index(["segmento", "métrica"]).round(4))
    if PROBA_BETO is not None:
        distinto = (proba >= 0.5) != (PROBA_BETO >= 0.5)
        acierta = ((proba >= 0.5) == y)[distinto].mean() if distinto.any() else np.nan
        print(f"Correlación con BETO título: {np.corrcoef(proba, PROBA_BETO)[0, 1]:.3f} | predicen distinto en "
              f"{distinto.sum():,} filas ({distinto.mean():.1%}); ahí acierta {nombre} el {acierta:.1%}")
    else:
        print("Sin finetune_titulo_oof.csv entre los datos de entrada: no se calcula la correlación con BETO título.")
    if MAX_FILAS:
        print(f"(prueba con {MAX_FILAS} filas: los números no son comparables con las referencias)")


def valores_beto(segmento, metrica):
    return REFERENCIAS["BETO título"][segmento][metrica]

## Corrida de todas las configuraciones

In [ ]:
RESULTADOS = {}
minutos_fold_48 = None  # minutos por fold medidos, llevados a 48 tokens
for cfg in CONFIGURACIONES:
    print(f"\n{'=' * 100}\nConfiguración {cfg['clave']}: texto {cfg['texto']}, max_length {cfg['max_length']}, "
          f"semilla {cfg['semilla']}", flush=True)
    hecho = nombre_completo(cfg)
    if hecho:
        print(f"Ya está completa ({hecho}): se saltea y se leen sus archivos.")
        RESULTADOS[cfg["clave"]] = {"nombre": hecho, "proba": pd.read_csv(archivos_finales(hecho)[0]).sort_values("fila")["proba_used"].to_numpy()}
        comparar(hecho, RESULTADOS[cfg["clave"]]["proba"])
        continue
    candidato, tokenizador = descargar(cfg)
    if candidato is None:
        print(f"Ningún candidato de la configuración {cfg['clave']} se pudo descargar: se saltea.")
        continue
    nombre, modelo = candidato["nombre"], candidato["modelo"]
    print(f"Modelo: {modelo} -> archivos finetune_{nombre}_*.csv")
    textos_train = armar_texto(train, cfg, modelo, tokenizador)
    textos_test = armar_texto(test, cfg, modelo, tokenizador)
    ids_train, mascara_train = tokenizar(tokenizador, textos_train, cfg["max_length"])
    ids_test, mascara_test = tokenizar(tokenizador, textos_test, cfg["max_length"])
    largo = mascara_train.sum(1)
    desconocidos = (ids_train == tokenizador.unk_token_id).sum().item() if tokenizador.unk_token_id is not None else 0
    print(f"Ejemplo: {textos_train[0]!r}\n  -> {tokenizador.convert_ids_to_tokens(ids_train[0][:largo[0]])}")
    print(f"Tokens por texto: mediana {largo.median().item():.0f}, máximo {largo.max().item()} | "
          f"cortados en {cfg['max_length']}: {(largo == cfg['max_length']).float().mean().item():.1%} | "
          f"tokens desconocidos: {desconocidos / largo.sum().item():.2%}", flush=True)

    resumen = {}
    for k in range(1, 6):
        archivo_oof, archivo_test, archivo_info = (os.path.join(RUTA_SALIDA, f"{nombre}_fold_{k}_{s}")
                                                   for s in ("oof.csv", "test.csv", "info.json"))
        if all(os.path.exists(a) for a in (archivo_oof, archivo_test, archivo_info)):
            print(f"Fold {k}: ya estaba guardado, se lee de {RUTA_SALIDA}")
            with open(archivo_info) as f:
                resumen[k] = json.load(f)
            continue
        print(f"--- Fold {k}: entrena con {np.sum(FOLD != k):,} filas, valida con {np.sum(FOLD == k):,}", flush=True)
        inicio = time.time()
        idx_val, mejor, historia = entrenar_fold(k, modelo, cfg["semilla"], ids_train, mascara_train, ids_test, mascara_test)
        proba_val = mejor["proba_val"]
        pd.DataFrame({"fila": idx_val, "fold": k, "proba_used": proba_val}).to_csv(archivo_oof, index=False)
        pd.DataFrame({"fila": np.arange(len(test)), "proba_used": mejor["proba_test"]}).to_csv(archivo_test, index=False)
        resumen[k] = {"accuracy": accuracy_score(y[idx_val], proba_val >= 0.5), "auc": roc_auc_score(y[idx_val], proba_val),
                      "log_loss": mejor["log_loss"], "mejor_epoca": mejor["epoca"], "minutos": (time.time() - inicio) / 60,
                      "historia": historia}
        with open(archivo_info, "w") as f:
            json.dump(resumen[k], f, indent=1)
        r = resumen[k]
        print(f"Fold {k}: accuracy {r['accuracy']:.4f} | AUC {r['auc']:.4f} | log loss {r['log_loss']:.4f} | "
              f"mejor época {r['mejor_epoca']} | {r['minutos']:.1f} min", flush=True)
        if minutos_fold_48 is None:
            minutos_fold_48 = r["minutos"] / factor(cfg)
            mostrar_estimacion(minutos_fold_48, f"Estimación con el primer fold ({r['minutos']:.1f} min)",
                               folds_hechos={cfg["clave"]: k})

    tabla = pd.DataFrame({k: {c: v for c, v in r.items() if c != "historia"} for k, r in resumen.items()}).T
    tabla.index.name = "fold"
    display(tabla.round(4))
    oof = pd.concat([pd.read_csv(os.path.join(RUTA_SALIDA, f"{nombre}_fold_{k}_oof.csv")) for k in range(1, 6)]).sort_values("fila")
    assert (oof["fila"].to_numpy() == np.arange(len(train))).all(), "Faltan filas en las predicciones out-of-fold"
    proba_test = np.mean([pd.read_csv(os.path.join(RUTA_SALIDA, f"{nombre}_fold_{k}_test.csv"))["proba_used"].to_numpy()
                          for k in range(1, 6)], axis=0)
    archivo_oof, archivo_test = archivos_finales(nombre)
    oof[["fila", "fold", "proba_used"]].to_csv(archivo_oof, index=False)
    pd.DataFrame({"fila": np.arange(len(test)), "proba_used": proba_test}).to_csv(archivo_test, index=False)
    print(f"Guardados {archivo_oof} ({len(oof):,} filas) y {archivo_test} ({len(test):,} filas) | "
          f"entrenamiento {tabla['minutos'].sum():.1f} min")
    RESULTADOS[cfg["clave"]] = {"nombre": nombre, "proba": oof["proba_used"].to_numpy()}
    comparar(nombre, RESULTADOS[cfg["clave"]]["proba"])
    del ids_train, mascara_train, ids_test, mascara_test

print(f"\nTiempo total de la corrida: {(time.time() - INICIO_CORRIDA) / 60:.1f} min")

## Resumen

Todas las configuraciones terminadas junto a las referencias. Los archivos a descargar son `finetune_{nombre}_oof.csv` y `finetune_{nombre}_test.csv` de cada configuración (van a `predicciones/finetune/` del proyecto).

In [ ]:
filas = {}
for referencia, valores in REFERENCIAS.items():
    filas[referencia] = {(s, m): valores[s][m] for s in MASCARAS for m in ("accuracy", "auc", "log_loss")}
for clave, resultado in RESULTADOS.items():
    filas[f"{clave}) {resultado['nombre']}"] = {(s, m): v for s, mascara in MASCARAS.items()
                                                for m, v in metricas(resultado["proba"], mascara).items()}
display(pd.DataFrame(filas).T.round(4))
for clave, resultado in RESULTADOS.items():
    print(f"{clave}) " + "  ".join(os.path.basename(a) for a in archivos_finales(resultado["nombre"])))
faltan = [cfg["clave"] for cfg in CONFIGURACIONES if cfg["clave"] not in RESULTADOS]
if faltan:
    print(f"Configuraciones sin resultado (falló la descarga de todos sus candidatos): {faltan}")